# Skull acoustic transparency at **500 kHz** — Colab

Time-reversal skull *transparency maps* + transducer placement, end to end, on a real benchmark skull, small enough for a free Colab GPU. A **manuscript-style figure at every step**.

**Pipeline:** `prepare` (skull map → solver inputs) → `sim outward --run` (full-wave GPU solve) → `extract` (→ Field Bundle) → `transparency` / `place`.

**Skull:** the ITRUSST transcranial-ultrasound benchmark skull (Aubry et al., *JASA* 2022), which is defined at 500 kHz. Its surface meshes are rasterized to homogeneous cortical bone (`c=2800`, water `1500`).

**Grid & runtime.** `dx = c0 / (f0·ppw)`, run at **6 PPW (~0.5 mm)** — benchmark-grade. Three things keep the skull affordable: the **surface-shell recorder** (`--recorder shell`) records only the calvarial field (no giant `genout_mod` dump — `genout.dat` stays sub-GB and `extract` never OOMs); the brain-center run is sized to the **head's own bounding box** (not a source-centered cube); and **`--truncate-mm`** trims the spine/lower mandible below the brain center (kept for the vault + suboccipital window). Together the 6 PPW solve needs **~6 GB GPU and ~6 GB system RAM**, so it **fits a free Colab T4** (16 GB GPU / ~12 GB RAM) in ~1-2 min. If you hit an out-of-memory, lower **`--surround-mm`** (or **`--truncate-mm`**) in step 2b; the resolution stays at 6 PPW.

## Setup

**Set the runtime to GPU first:** *Runtime ▸ Change runtime type ▸ GPU (T4)*, then run the cells top to bottom.

The cell below installs `skull_transparency` and `trimesh`, fetches the CUDA full-wave solver binary, and reports the GPU it found.

In [ ]:
# --- setup: install the package + fetch the CUDA solver binary + mesh tooling ---
import os, sys, glob, shutil, subprocess
IN_COLAB = "google.colab" in sys.modules

# skull_transparency (pinned to the colab-notebook branch: the non-cubic grid + --surround-mm/
# --truncate-mm live there; drop @colab-notebook once merged to main) + trimesh (rasterize the STL).
# --force-reinstall --no-deps so a cached older build in a warm runtime is actually REPLACED with
# the branch version (base deps are just numpy/scipy, already in Colab).
!pip -q install --force-reinstall --no-deps "git+https://github.com/pinton-lab/skull_transparency.git@colab-notebook"
!pip -q install trimesh
# fail loudly HERE (not in 2b) if a stale package is still active
_help = subprocess.run(["skull-transparency", "prepare", "--help"], capture_output=True, text=True).stdout
assert "--truncate-mm" in _help, ("A stale skull_transparency is active. Fix: Runtime > Disconnect and "
    "delete runtime, then reopen this notebook FROM GITHUB (so the cells are the branch version) and Run all.")
print("skull_transparency (colab-notebook branch) ready -- prepare has --truncate-mm")

# The full-wave CUDA solver binary ships in the fullwave2-ultra repo.
# Clone it and point FULLWAVE2_BIN at it.
!git clone --depth 1 https://github.com/pinton-lab/fullwave2-ultra.git /content/fullwave2-ultra 2>/dev/null || true
hits = glob.glob("/content/fullwave2-ultra/**/bench_3d_opt", recursive=True)   # don't hardcode the path
assert hits, "bench_3d_opt not found in the clone -- re-run this cell (network?) or check the repo."
os.environ["FULLWAVE2_BIN"] = hits[0]
os.chmod(hits[0], 0o755)                                                       # git may drop the +x bit
print("solver:", os.environ["FULLWAVE2_BIN"], "| executable:", os.access(hits[0], os.X_OK))

# GPU report -- the Section-2 solve needs ~6 GB (6 PPW, bbox grid + truncation); a free T4 fits.
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True)
if gpu.returncode == 0 and gpu.stdout.strip():
    print("GPU:", gpu.stdout.strip(), "\n(the 6-PPW solve needs ~6 GB -> a free T4 is enough; if OOM, lower --surround-mm/--truncate-mm in 2b)")
else:
    print("NO GPU visible -- Runtime > Change runtime type > GPU (a free T4 is enough) before Section 2.")

## Figure helpers (manuscript style)

One reusable set of multi-panel figures, used at each step below. They work on both the no-GPU synthetic bundle (Section 1) and the real solved bundle (Section 2).

In [ ]:
import numpy as np, matplotlib.pyplot as plt

INK, EDGE, FOCUS = "#1d1d1f", "#2c6fbf", "#c0392b"
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 10, "figure.facecolor": "white",
                     "axes.facecolor": "white", "axes.titlesize": 10, "savefig.dpi": 120})

def _ortho(ax3, vol, dx, cmap, vmin, vmax, log=False, titles=("sagittal","coronal","axial")):
    v = np.asarray(vol, float); ctr = np.array(v.shape) // 2
    d = np.log10(np.maximum(v, (v[v>0].min() if (v>0).any() else 1))) if log else v
    lo = np.log10(vmin) if (log and vmin>0) else vmin
    hi = np.log10(vmax) if (log and vmax>0) else vmax
    for a, (s, t) in zip(ax3, zip([d[ctr[0]].T, d[:,ctr[1]].T, d[:,:,ctr[2]].T], titles)):
        im = a.imshow(s, origin="lower", cmap=cmap, vmin=lo, vmax=hi, aspect="equal",
                      extent=[0, s.shape[1]*dx, 0, s.shape[0]*dx])
        a.set_title(t); a.set_xlabel("mm")
    ax3[0].set_ylabel("mm"); return im

def fig_skull(c, dx, title="Skull sound-speed map  c(x)"):
    fig, ax = plt.subplots(1, 3, figsize=(13, 4.3), constrained_layout=True)
    im = _ortho(ax, c, dx, "bone", 1500, float(np.max(c)))
    fig.colorbar(im, ax=ax, fraction=0.02, pad=0.01, label="c (m/s)")
    fig.suptitle(title, fontweight="bold"); plt.show()


# -- 3D surface renders (matplotlib mplot3d point cloud): the same picture the napari report
#    renders show -- the calvarial surface patches in 3D perspective, coloured per patch. --
_VIEWS = ((18, -60), (18, 60), (78, -90))            # left-3/4, right-3/4, top ("turn the skull")
_AXL = ("x  L-R (mm)", "y  P-A (mm)", "z  I-S (mm)")

def _db(val):
    # transparency amplitude in dB re the 98th-pct patch (-40 dB floor) -- the whole-skull convention
    amp = np.sqrt(np.maximum(np.asarray(val, float), 0.0))
    ref = np.percentile(amp, 98.0) or (amp.max() or 1.0)
    return 20.0 * np.log10(np.maximum(amp / ref, 10.0 ** (-40.0 / 20.0)))

def _style3d(ax, P, elev, azim):
    ax.view_init(elev=elev, azim=azim); ax.set_box_aspect((1, 1, 1))
    lo, hi = P.min(0), P.max(0); mid = (hi + lo) / 2; s = (hi - lo).max() / 2 or 1.0
    for k, m, lab in zip("xyz", mid, _AXL):
        getattr(ax, f"set_{k}lim")(m - s, m + s); getattr(ax, f"set_{k}label")(lab, fontsize=7, labelpad=-8)
    for pane in (ax.xaxis, ax.yaxis, ax.zaxis):
        pane.pane.set_facecolor((1, 1, 1, 0)); pane.pane.set_edgecolor((0, 0, 0, 0.08))
        pane.set_ticklabels([]); pane.set_tick_params(length=0)
    ax.grid(False)

def _render3d(fig, gs_slices, P, c, cmap, clabel, vmin, vmax, s=6):
    sc = None
    for gs, (elev, azim) in zip(gs_slices, _VIEWS):
        ax = fig.add_subplot(gs, projection="3d")
        sc = ax.scatter(P[:, 0], P[:, 1], P[:, 2], c=c, cmap=cmap, vmin=vmin, vmax=vmax,
                        s=s, linewidths=0, depthshade=True)
        _style3d(ax, P, elev, azim)
    cb = fig.colorbar(sc, ax=fig.axes, fraction=0.014, pad=0.0); cb.set_label(clabel, fontsize=9)
    return sc

def fig_surface(t, dx, vals, title, cmap="inferno", label="value"):
    P = np.asarray(t.surf_vox, float) * dx; vn = np.asarray(vals, float) / (np.max(vals) or 1)
    fig = plt.figure(figsize=(13.5, 5)); gs = fig.add_gridspec(1, 3)
    _render3d(fig, [gs[0, i] for i in range(3)], P, vn, cmap, label, 0.0, 1.0)
    fig.suptitle(title, fontsize=13, fontweight="bold"); plt.show()

def fig_transparency(t, dx, title="Skull transparency map"):
    P = np.asarray(t.surf_vox, float) * dx
    disp = _db(t.value); vlo = max(float(np.percentile(disp, 2.0)), -40.0)
    vn = np.maximum(t.value, 0) / (np.max(t.value) or 1)
    fig = plt.figure(figsize=(13.5, 8)); gs = fig.add_gridspec(2, 3, height_ratios=[1.5, 1])
    _render3d(fig, [gs[0, i] for i in range(3)], P, disp, "inferno",
              "transparency amplitude (dB re 98th pct)", vlo, 0.0)
    b0 = fig.add_subplot(gs[1, 0]); b0.hist(vn, bins=40, color=EDGE, alpha=0.85)
    b0.set_title("transparency histogram", fontsize=10); b0.set_xlabel("normalized"); b0.set_ylabel("patches")
    b1 = fig.add_subplot(gs[1, 1]); b1.scatter(np.asarray(t.rad_mm), vn, s=5, color=EDGE, alpha=0.4)
    b1.set_title("transparency vs source distance", fontsize=10); b1.set_xlabel("distance (mm)")
    b2 = fig.add_subplot(gs[1, 2]); b2.axis("off")
    b2.text(0.5, 0.5, f"{len(P)} surface patches\nbright = transparent bone\n\n"
            f"source depth {np.median(t.rad_mm):.0f} mm (median)", ha="center", va="center",
            bbox=dict(boxstyle="round,pad=0.6", fc="#f4f6f9", ec=EDGE))
    fig.suptitle(title, fontsize=13, fontweight="bold"); plt.show()

def fig_placement(t, pl, dx, title="Transducer placement"):
    P = np.asarray(t.surf_vox, float) * dx
    foot = np.zeros(len(P), bool); foot[np.asarray(pl.footprint_surf_idx)] = True
    fig = plt.figure(figsize=(13.5, 5)); gs = fig.add_gridspec(1, 3)
    for gsi, (elev, azim) in zip([gs[0, i] for i in range(3)], _VIEWS):
        ax = fig.add_subplot(gsi, projection="3d")
        ax.scatter(*P[~foot].T, c="#c9ced6", s=4, linewidths=0, depthshade=True)    # skull (context)
        ax.scatter(*P[foot].T, c=FOCUS, s=20, linewidths=0, depthshade=True)        # bowl footprint
        _style3d(ax, P, elev, azim)
    fig.suptitle(f"{title}   —   footprint {pl.n_footprint_patches} patches, incidence "
                 f"{pl.incidence_deg:.1f} deg, transparency score {pl.transparency_score:.3f}",
                 fontsize=12, fontweight="bold"); plt.show()

print("figure helpers ready")

## 1 · Quick taste — no GPU, no data

The analysis + figures on a shipped **synthetic** Field Bundle, so you see the whole thing before the GPU solve.

In [ ]:
import skull_transparency as st
b0 = st.load_bundle(st.make_synthetic_bundle("synthetic_bundle"))
t0 = st.compute_transparency_map(b0)
pl0 = st.place_bowl(t0, st.BowlConstraints(focal_length_mm=60.0, bowl_radius_mm=15.0, theta_max_deg=35.0))
print(f"{len(t0.surf_vox)} surface patches | placement incidence {pl0.incidence_deg:.1f} deg, "
      f"score {pl0.transparency_score:.3f}")
dx0 = b0.grid["dx_m"] * 1e3
fig_transparency(t0, dx0, title="Transparency map (synthetic, no GPU)")
fig_placement(t0, pl0, dx0, title="Placement (synthetic, no GPU)")

## 2 · Real 500 kHz simulation on the ITRUSST benchmark skull

### 2a · Build the skull sound-speed map  — *figure: input skull*

Downloads the benchmark's inner/outer surface meshes (~13 MB) and rasterizes them to the grid as homogeneous cortical bone at **6 PPW (~0.5 mm)**. Set `USE_REAL_SKULL = False` for an offline toy shell.

In [ ]:
import os, urllib.request, numpy as np

PPW = 6                        # 6 PPW (~0.5 mm) = benchmark-grade (do not lower). Shell recorder keeps the
                               # OUTPUT sub-GB; bbox grid + spine/mandible truncation keep the solve at
                               # ~6 GB GPU/RAM at 6 PPW -> fits a free Colab T4 (lower --surround-mm if OOM).
USE_REAL_SKULL = True
C0, F0 = 1500.0, 500e3
pitch = round(C0 / (F0 * PPW) * 1e3, 3)     # mm/voxel to match the sim grid

if USE_REAL_SKULL:
    import trimesh
    BASE = ("https://raw.githubusercontent.com/ucl-bug/transcranial-ultrasound-benchmarks"
            "/master/intercomparison/skull-stl")
    for f in ("skull_inner.stl", "skull_outer.stl"):
        if not os.path.exists(f):
            urllib.request.urlretrieve(f"{BASE}/{f}", f)
    outer = trimesh.load("skull_outer.stl"); inner = trimesh.load("skull_inner.stl")
    lo = outer.bounds[0] - 2*pitch
    dims = np.ceil((outer.bounds[1] + 2*pitch - lo) / pitch).astype(int)
    def _raster(mesh):
        vg = mesh.voxelized(pitch=pitch).fill(); idx = np.floor((vg.points - lo) / pitch).astype(int)
        g = np.zeros(dims, bool); ok = np.all((idx >= 0) & (idx < dims), axis=1); idx = idx[ok]
        g[idx[:,0], idx[:,1], idx[:,2]] = True; return g
    bone = _raster(outer) & ~_raster(inner)
    c = np.where(bone, 2800.0, C0).astype(np.float32)
    affine = np.diag([pitch, pitch, pitch, 1.0]); affine[:3, 3] = lo
    print(f"ITRUSST skull @ {pitch} mm: {tuple(int(d) for d in dims)} = {c.size/1e6:.1f} M voxels; "
          f"bone {bone.sum()*pitch**3/1000:.0f} cm^3")
else:
    n = int(round(120 * 0.5 / pitch))
    c = np.full((n, n, n), C0, np.float32)
    zz, yy, xx = np.mgrid[0:n, 0:n, 0:n]
    r = np.sqrt((xx-n/2)**2 + (yy-n/2)**2 + (zz-n/2)**2) * pitch
    c[(r > 40) & (r < 46)] = 2600.0
    affine = np.diag([pitch, pitch, pitch, 1.0]); affine[:3, 3] = -pitch * n / 2
    print("toy shell:", c.shape)

np.save("head_c.npy", c); np.save("affine.npy", affine)
import json; json.dump({"preset": "ctx500", "f0_hz": F0, "ppw": PPW}, open("ctx500.json", "w"))
fig_skull(c, pitch, title=f"Input skull @ {pitch} mm  ({'ITRUSST benchmark' if USE_REAL_SKULL else 'toy shell'})")

### 2b · `prepare` — skull map → solver inputs (no GPU)

Seats an omnidirectional source at the brain center (`--center`) for the whole-skull transparency map, and writes the `.dat` solver deck at `dx = c0/(f0·ppw)`. The grid is sized to the **head's own bounding box** (not a source-centered cube), and two knobs shrink it further:

- **`--surround-mm`** — water margin around the head (smaller = smaller grid).
- **`--truncate-mm`** — trims the head along the inferior-superior axis to this many mm below the brain center, cutting the **spine and lower mandible** (not cranial windows). Keeps the whole vault and the suboccipital region; drops the tall spine tail that otherwise dominates the grid. Set `TRUNCATE_MM = None` for the whole head.

In [ ]:
SURROUND_MM = 12      # water margin (mm) around the head -> sizes the grid. Lower (e.g. 8) if 2c OOMs.
TRUNCATE_MM = 75      # keep <=75 mm below the brain center along S-I: drops the spine/lower mandible
                      # (well below the suboccipital window). None = whole head.
_trunc = f"--truncate-mm {TRUNCATE_MM}" if TRUNCATE_MM else ""
!skull-transparency prepare --c-map head_c.npy --affine affine.npy --center --transducer ctx500.json --surround-mm {SURROUND_MM} {_trunc} --out run
import json; m = json.load(open("run/meta.json"))
gs = m.get("grid_shape", [m['N']] * 3)   # sized to the head bbox (non-cubic), +48 absorbing layer/side
print(f"sim grid {gs[0]}x{gs[1]}x{gs[2]} (solver +48 absorbing layer/side)  dx={m['dX_m']*1e3:.3f} mm  F0={m['F0']/1e3:.0f} kHz")

### 2c · Outward full-wave solve (GPU)

Runs the CUDA solver with `--recorder shell` — it records the field only on the calvarial surface (all the transparency map needs), so there is **no** `genout_mod` volume dump and `genout.dat` stays sub-GB. About 1-2 min on a GPU at 6 PPW. *(The field figures come after `extract`, in 2d.)*

In [ ]:
import os, sys, subprocess
# capture the solver output so a failure is visible and DIAGNOSED here (not swallowed)
proc = subprocess.run([sys.executable, "-m", "skull_transparency.sim", "outward",
                       "--sim", "run", "--out", "run", "--run", "--recorder", "shell"],
                      capture_output=True, text=True)
go = "run/outward/genout.dat"
have = os.path.exists(go) and os.path.getsize(go) > 0
if proc.returncode != 0 or not have:
    rc = proc.returncode
    print("solver exit code:", rc, (f"(killed by signal {-rc})" if rc < 0 else ""))
    print("run/outward contents:", sorted(os.listdir("run/outward")) if os.path.isdir("run/outward") else "MISSING")
    print("===== solver STDOUT (tail) =====\n" + (proc.stdout[-3000:] or "(empty)"))
    print("===== solver STDERR (tail) =====\n" + (proc.stderr[-3000:] or "(empty)"))
    if rc < 0:
        msg = (f"Process KILLED BY SIGNAL {-rc} with no error output -> the host (system) RAM out-of-\n"
               "memory killer, NOT a GPU error. Deck prep at 6 PPW needs ~6 GB (bbox grid + truncation),\n"
               "which normally fits a free Colab runtime; if yours has less free RAM it can still be\n"
               "killed. Fixes (keep PPW=6): lower --surround-mm or --truncate-mm in cell 2b, or use a\n"
               "High-RAM / L4 / A100 runtime (Runtime > Change runtime type).")
    elif rc > 0:
        msg = (f"Solver exited {rc} -> it crashed. Usual causes: GPU runtime OFF (Runtime > Change\n"
               "runtime type > GPU); CUDA out-of-memory (see the STDERR above -> use a bigger GPU or\n"
               "lower --surround-mm in 2b); or a missing binary (re-run the setup cell).")
    else:   # exited 0 but wrote no genout.dat
        msg = ("Solver exited 0 but wrote no genout.dat -> it ran but produced nothing, usually too\n"
               "little GPU memory for the grid (the STDOUT above reports how much it needed).\n"
               "Fix: shrink the grid (cell 2b --surround-mm) or use a bigger GPU. Keep PPW=6.")
    raise RuntimeError(msg)
assert not os.path.exists("run/outward/genout_mod.dat"), \
    "unexpected genout_mod.dat (the shell recorder should not write one)"
print(f"OK -- genout.dat (surface shell): {os.path.getsize(go)/1e6:.0f} MB   |   genout_mod: none")

### 2d · Extract → Field Bundle → transparency map  — *figures: outward surface field + transparency*

The shell recorder captures the field **on the calvarial surface** (no volume dump), so the field figures are surface quantities shown on the skull cross-section: first the raw peak pressure reaching the bone, then the transparency map derived from it.

In [ ]:
!skull-transparency extract --run run/outward --sim run --out run/bundle

import skull_transparency as st
b = st.load_bundle("run/bundle")
dx = b.grid["dx_m"]*1e3
t = st.compute_transparency_map(b)

fig_surface(t, dx, t.Pmax, title="Outward field @ 500 kHz — peak pressure at the calvaria (ITRUSST skull)",
            cmap="inferno", label="normalized peak pressure")
fig_transparency(t, dx, title="Skull transparency @ 500 kHz  (ITRUSST benchmark skull)")

### 2e · Place a focused window on the map  — *figure: placement*

In [ ]:
pl = st.place_bowl(t, st.BowlConstraints(focal_length_mm=60.0, bowl_radius_mm=32.0, theta_max_deg=35.0))
print(f"incidence {pl.incidence_deg:.1f} deg, transparency score {pl.transparency_score:.3f}")
fig_placement(t, pl, dx, title="Transducer placement on the transparency map")

## Interactive placement (desktop)

Section 2e picks the window programmatically. To place it by hand on the 3-D transparency map, use the napari viewer — napari needs a display, so run it on your own machine rather than in Colab. After a solve, `skull-transparency position --bundle run/bundle --interactive` opens the skull surface, the transparency coupling, and the chosen bowl in 3-D; install the viewer with `pip install "skull_transparency[viz]"`. A ready-to-run example is [`examples/brain_center/view_braincenter_napari.py`](https://github.com/pinton-lab/skull_transparency/blob/colab-notebook/examples/brain_center/view_braincenter_napari.py).